In [1]:
import pandas as pd
import pandas as pd
from sklearn.feature_selection import mutual_info_regression
import pandas as pd
import numpy as np
import time
import asyncio
import httpx
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import time
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.display.max_columns = None


df = pd.read_csv('xydone.csv')

In [2]:
print(df.shape)

(1557121, 15)


In [8]:
df.head()

,Købesum,Vær.,Byggeår,dato,stype,m2,btype,addresse,postnummer,by,område,region
0,1398250,4,1956,27-05-2025,Fam. Salg,83.0,Fritidshus,"Nordmandsvej 12, Lyngsbæk",8400,Ebeltoft,Øst- og Midtjylland,Jylland
1,2620000,5,1972,27-05-2025,Alm. Salg,120.0,Rækkehus,Rosenlyparken 80,2670,Greve,"Hovedstaden, København",Sjælland
2,6450000,6,1906,27-05-2025,Alm. Salg,240.0,Landejendom,Karlslunde Centervej 76,4030,Tune,Andre øer,Sjælland
3,460000,8,1908,27-05-2025,Andet,176.0,Villa,Hovedgaden 42,8763,Rask Mølle,Øst- og Midtjylland,Jylland
4,2951000,5,1974,27-05-2025,Alm. Salg,143.0,Villa,Kirkebakken 66,4621,Gadstrup,Andre øer,Sjælland


In [2]:
dfl = df[df['btype'] != 'Ejerlejlighed']

print(f"Shape after filtering: {dfl.shape}")



Shape after filtering: (1217313, 15)


In [ ]:
import pandas as pd
import requests
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import os

# Configuration
MAX_RETRIES = 4
INITIAL_BACKOFF_DELAY = 2
CHECKPOINT_INTERVAL = 1000
CHECKPOINT_FILENAME = "bfe_checkpoint_not_lejligheder.csv" # Changed for safety
MAX_WORKERS = 10 # Number of concurrent requests

def get_bfe_for_non_apartment(address_str, postnr, username, password):
    """
    Fetches the BFE number for non-apartment properties (houses, commercial, etc.)
    Uses only the building-level endpoint since these aren't individual units.
    """
    full_address_query = f"{address_str}, {postnr}"
    current_delay = INITIAL_BACKOFF_DELAY

    for attempt in range(MAX_RETRIES):
        try:
            # 1. Get the unique address ID from DAWA
            dawa_url = "https://api.dataforsyningen.dk/adresser"
            dawa_params = {'q': full_address_query, 'struktur': 'mini'}
            dawa_resp = requests.get(dawa_url, params=dawa_params, timeout=30)
            dawa_resp.raise_for_status()
            dawa_data = dawa_resp.json()

            if not dawa_data:
                return None

            adresse_id = dawa_data[0].get('id')
            if not adresse_id:
                return None

            # 2. Get BFE number using ONLY the building-level endpoint
            building_url = "https://services.datafordeler.dk/DAR/DAR_BFE_Public/1/rest/adresseTilBygningBfe"
            bfe_params = {
                "adresseId": adresse_id,  # Use the address ID, not the DAWA params
                "username": username,
                "password": password
            }
            building_resp = requests.get(building_url, params=bfe_params, timeout=30)
            building_resp.raise_for_status()
            building_data = building_resp.json()

            if building_data and isinstance(building_data, list):
                return building_data[0]

            return None
            
        except (requests.RequestException, requests.Timeout) as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(current_delay)
                current_delay *= 2
            else:
                return None
    return None

def process_address_with_index(args):
    """Wrapper function to process a single address with its index"""
    index, row, username, password = args
    # Changed function call to the new apartment-specific function
    result = get_bfe_for_non_apartment(row['addresse'], row['postnummer'], username, password)
    return index, result

def find_bfe_numbers_parallel(df_full, df_to_process, username, password):
    print(f"🚀 Processing {len(df_to_process)} addresses with {MAX_WORKERS} workers...")
    processed_count = 0
    success_count = 0
    task_args = [(index, row, username, password)
                 for index, row in df_to_process.iterrows()]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_args = {executor.submit(process_address_with_index, args): args
                          for args in task_args}
        pbar = tqdm(as_completed(future_to_args), total=len(task_args), desc="Processing")
        for future in pbar:
            try:
                index, result = future.result()
                df_full.loc[index, 'bfe_nummer'] = result
                processed_count += 1
                if result is not None:
                    success_count += 1
                if processed_count > 0:
                    success_rate = (success_count / processed_count) * 100
                    pbar.set_postfix_str(f"Success Rate: {success_rate:.1f}%")
                if processed_count % CHECKPOINT_INTERVAL == 0:
                    print(f"\n--- Checkpoint reached. Saving progress to {CHECKPOINT_FILENAME}... ---")
                    df_full.to_csv(CHECKPOINT_FILENAME)
            except Exception as e:
                print(f"Error processing task: {e}")
                continue

    print(f"\n--- Final save to {CHECKPOINT_FILENAME}... ---")
    df_full.to_csv(CHECKPOINT_FILENAME)
    print("✅ Done. All progress saved.")
    return df_full

def main():
    service_user_name = "XEVPPQIYSU"
    service_user_password = "Luffygear3!"

    if os.path.exists(CHECKPOINT_FILENAME):
        print(f"✅ Checkpoint file '{CHECKPOINT_FILENAME}' found. Resuming progress.")
        df_full = pd.read_csv(CHECKPOINT_FILENAME, index_col=0)
        # Find the last successfully processed index
        last_success_idx = df_full[df_full['bfe_nummer'].notna()].index.max()
        if pd.notna(last_success_idx):
            # Process from the next batch after the last success
            start_idx = last_success_idx + 1
            df_to_process = df_full.iloc[start_idx:][df_full.iloc[start_idx:]['bfe_nummer'].isnull()]
        else:
            # No successes yet, process all nulls
            df_to_process = df_full[df_full['bfe_nummer'].isnull()]
    else:
        print("ℹ️ No checkpoint file found. Starting from scratch.")
        df_full = dfl.copy()
        df_full['bfe_nummer'] = None
        df_to_process = df_full[df_full['bfe_nummer'].isnull()]

    # REMOVE THIS LINE - it's overriding your logic above!
    # df_to_process = df_full[df_full['bfe_nummer'].isnull()]

    if df_to_process.empty:
        print("✅ Processing is already complete according to the checkpoint file.")
    else:
        df_final = find_bfe_numbers_parallel(
            df_full=df_full,
            df_to_process=df_to_process,
            username=service_user_name,
            password=service_user_password,
        )
        print("\n--- Final Results Sample ---")
        print(df_final.head())
        print(f"\nFinal Success Rate: {df_final['bfe_nummer'].notna().mean() * 100:.1f}%")

# Run it
main()

ℹ️ No checkpoint file found. Starting from scratch.
🚀 Processing 1217313 addresses with 10 workers...


In [ ]:
# Load both datasets
df_checkpoint = pd.read_csv('bfe_checkpoint_not_lejligheder.csv', index_col=0)
df_sales = pd.read_csv('sales_data3.csv')
df_checkpoint['bfe_nummer'] = df_checkpoint['bfe_nummer'].dropna().astype(int)
df_sales['bfe_nummer'] = df_sales['bfe_nummer'].dropna().astype(int)
print(f"Checkpoint data shape: {df_checkpoint.shape}")
print(f"Sales data shape: {df_sales.shape}")

# Check the BFE number columns
print(f"\nCheckpoint BFE numbers - non-null: {df_checkpoint['bfe_nummer'].notna().sum()}")
print(f"Sales BFE numbers - non-null: {df_sales['bfe_nummer'].notna().sum()}")

# Join them on bfe_nummer
df_joined = df_checkpoint.merge(
    df_sales, 
    on='bfe_nummer', 
    how='inner',  # Only keep rows where both have bfe_nummer
    suffixes=('_checkpoint', '_sales')
)
df_joined = df_joined.drop_duplicates()
unique_bfe_count = df_joined['bfe_nummer'].nunique()
print(f"\nUnique BFE numbers in joined data: {unique_bfe_count:,}")


print(f"\nJoined data shape: {df_joined.shape}")
print(f"Successful matches: {len(df_joined):,}")

# Show sample of joined data
print("\nSample of joined data:")
print(df_joined.head())

# Save the joined data
df_joined.to_csv('joined_bfe_datanew.csv', index=False)
print("\n✅ Saved joined data to 'joined_bfe_data.csv'")

Checkpoint data shape: (339808, 16)
Sales data shape: (2950849, 6)

Checkpoint BFE numbers - non-null: 191645
Sales BFE numbers - non-null: 2950849

Unique BFE numbers in joined data: 99,483

Joined data shape: (295044, 21)
Successful matches: 295,044

Sample of joined data:
   Købesum  Vær.  Byggeår dato_checkpoint      stype    m2          btype  \
0  4900000     3     1911      27-05-2025  Alm. Salg  82.0  Ejerlejlighed   
1  4900000     3     1911      27-05-2025  Alm. Salg  82.0  Ejerlejlighed   
2  4900000     3     1911      27-05-2025  Alm. Salg  82.0  Ejerlejlighed   
3  4900000     3     1911      27-05-2025  Alm. Salg  82.0  Ejerlejlighed   
4  4900000     3     1911      27-05-2025  Alm. Salg  82.0  Ejerlejlighed   

                  addresse  postnummer           by                  område  \
0  Nordlandsgade 8, st. th        2300  København S  Hovedstaden, København   
1  Nordlandsgade 8, st. th        2300  København S  Hovedstaden, København   
2  Nordlandsgade 8, st. 

In [7]:
# Load the checkpoint data
df_checkpoint = pd.read_csv('bfe_checkpoint_lejligheder.csv', index_col=0)

# Get first 1000 records
first_1000 = df_checkpoint.head(1000).copy()

# Split into successful and failed
successful = first_1000[first_1000['bfe_nummer'].notna()].copy()
failed = first_1000[first_1000['bfe_nummer'].isna()].copy()

print(f"First 1000 records analysis:")
print(f"Total: {len(first_1000):,}")
print(f"Successful: {len(successful):,} ({len(successful)/len(first_1000)*100:.1f}%)")
print(f"Failed: {len(failed):,} ({len(failed)/len(first_1000)*100:.1f}%)")

print(f"\n{'='*50}")
print("FAILED ADDRESSES - Let's see what's different:")
print(f"{'='*50}")

# Show sample of failed addresses
print("\nSample failed addresses:")
print(failed[['addresse', 'postnummer']].head(10))

print(f"\n{'='*50}")
print("SUCCESSFUL ADDRESSES - For comparison:")
print(f"{'='*50}")

# Show sample of successful addresses
print("\nSample successful addresses:")
print(successful[['addresse', 'postnummer', 'bfe_nummer']].head(10))

print(f"\n{'='*50}")
print("PATTERN ANALYSIS:")
print(f"{'='*50}")

# Check for patterns in failed addresses
print("\nFailed addresses - postal code distribution:")
print(failed['postnummer'].value_counts().head(10))

print("\nSuccessful addresses - postal code distribution:")
print(successful['postnummer'].value_counts().head(10))

# Check for address format patterns
print(f"\n{'='*30}")
print("ADDRESS FORMAT ANALYSIS:")
print(f"{'='*30}")

def analyze_address_format(df, label):
    print(f"\n{label}:")
    # Check for various patterns
    has_comma = df['addresse'].str.contains(',', na=False).sum()
    has_floor = df['addresse'].str.contains(r'\d+\.\s*(tv|th|mf)', na=False).sum()
    has_number = df['addresse'].str.contains(r'\d+', na=False).sum()
    
    print(f"  Contains comma: {has_comma}/{len(df)} ({has_comma/len(df)*100:.1f}%)")
    print(f"  Contains floor info (tv/th/mf): {has_floor}/{len(df)} ({has_floor/len(df)*100:.1f}%)")
    print(f"  Contains numbers: {has_number}/{len(df)} ({has_number/len(df)*100:.1f}%)")
    
    # Show some examples
    print(f"  Examples:")
    for addr in df['addresse'].head(5):
        print(f"    '{addr}'")

analyze_address_format(failed, "FAILED ADDRESSES")
analyze_address_format(successful, "SUCCESSFUL ADDRESSES")

print(f"\n{'='*50}")
print("RECOMMENDATIONS:")
print(f"{'='*50}")
print("Look at the patterns above to see:")
print("1. Are failed addresses missing apartment info (floor/side)?")
print("2. Are they in specific postal areas?")
print("3. Do they have different formatting?")
print("4. Are they perhaps not real addresses or have typos?")

First 1000 records analysis:
Total: 1,000
Successful: 756 (75.6%)
Failed: 244 (24.4%)

FAILED ADDRESSES - Let's see what's different:

Sample failed addresses:
                             addresse  postnummer
22                Algade 25, 1., Ejby        5592
26         Vestre Bakkevej 40, st. tv        8920
37  Albanivej 18, st., Nørre Lyndelse        5792
40         Vestre Bakkevej 40, st. th        8920
44             Hammerlodden 19, 1. th        4800
62        Hans Tausens Gade 17, 3. th        5000
65        Hans Tausens Gade 17, 1. th        5000
67        Hans Tausens Gade 17, 2. tv        5000
73                Møllergade 104D, 2.        5700
76        Hans Tausens Gade 17, 1. tv        5000

SUCCESSFUL ADDRESSES - For comparison:

Sample successful addresses:
                          addresse  postnummer  bfe_nummer
6          Prins Valdemars Alle 9B        3450    249142.0
7          Nordlandsgade 8, st. th        2300    134089.0
10        Horsekildevej 32, st. th        2

/var/folders/qr/zdl1jn496pg73658gxs99sxh0000gn/T/ipykernel_89167/1964470605.py:52: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_floor = df['addresse'].str.contains(r'\d+\.\s*(tv|th|mf)', na=False).sum()


In [ ]:
# Let's test some of the failed addresses manually to understand the issues
failed_sample = [
    "Algade 25, 1., Ejby, 5592",
    "Vestre Bakkevej 40, st. tv, 8920", 
    "Albanivej 18, st., Nørre Lyndelse, 5792"
]

# Test what DAWA returns for these
import requests

def test_address_lookup(address_query):
    dawa_url = "https://api.dataforsyningen.dk/adresser"
    dawa_params = {'q': address_query, 'struktur': 'mini'}
    
    try:
        resp = requests.get(dawa_url, params=dawa_params, timeout=30)
        data = resp.json()
        print(f"Query: '{address_query}'")
        print(f"Results: {len(data)} found")
        if data:
            print(f"First result: {data[0]}")
        else:
            print("No results found!")
        print("-" * 50)
    except Exception as e:
        print(f"Error: {e}")

for addr in failed_sample:
    test_address_lookup(addr)

Query: 'Algade 25, 1., 5592'
Results: 1 found
First result: {'id': '0a3f50b2-83f7-32b8-e044-0003ba298018', 'status': 1, 'darstatus': 3, 'vejkode': '0027', 'vejnavn': 'Algade', 'adresseringsvejnavn': 'Algade', 'husnr': '25', 'etage': '1', 'dør': None, 'supplerendebynavn': 'Ejby', 'postnr': '5592', 'postnrnavn': 'Ejby', 'stormodtagerpostnr': None, 'stormodtagerpostnrnavn': None, 'kommunekode': '0410', 'adgangsadresseid': '0a3f5087-fd71-32b8-e044-0003ba298018', 'x': 9.92602262, 'y': 55.42783349, 'href': 'https://api.dataforsyningen.dk/adresser/0a3f50b2-83f7-32b8-e044-0003ba298018', 'betegnelse': 'Algade 25, 1., Ejby, 5592 Ejby'}
--------------------------------------------------
Query: 'Vestre Bakkevej 40, st. tv, 8920'
Results: 0 found
No results found!
--------------------------------------------------
Query: 'Albanivej 18, st., Nørre Lyndelse, 5792'
Results: 1 found
First result: {'id': '0a3f50b6-2223-32b8-e044-0003ba298018', 'status': 1, 'darstatus': 3, 'vejkode': '0015', 'vejnavn': '